In [1]:
!pip install -U bitsandbytes accelerate peft transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 13.2 MB/s eta 0:00:00


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = "/content/drive/MyDrive/llama_models/llama3_2_2b"
ADAPTER_DIR = "/content/drive/MyDrive/llama_stage1_marathi_qlora/checkpoint-702"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, local_files_only=True)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    local_files_only=True,
    trust_remote_code=True
)

model = PeftModel.from_pretrained(model, ADAPTER_DIR)
model.eval()

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 2048)
        (layers): ModuleList(
          (0-15): 16 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

In [3]:
prompt1 = "भारतामध्ये शिक्षण व्यवस्था ही"

In [4]:
prompt2 = "कोरोना महामारीमुळे समाजावर"

In [5]:
prompt3 = "एका गावात एक शेतकरी राहत होता."

In [8]:
inputs = tokenizer(prompt3, return_tensors="pt").to(model.device)

out = model.generate(
    **inputs,
    max_new_tokens=250,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

print(tokenizer.decode(out[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


एका गावात एक शेतकरी राहत होता. त्याचे घर सगळीकडे केवळ बाजारबाजार होते. परंतु, एका दिवशी, विमानांच्या जोरावर या गावाला एक अनोखी घटना घडली. एरोनेट स्टोर्समध्ये ७५ लाख प्लास्टिक पेअर्स आणि ३० लाख रुपये भाडे असे सर्वांत मोठे पैकीच विमानतळावरून आढळले. यामुळेच या गावातील दुसऱ्या वर्षी नुकसान झालेल्या शेतकऱ्यांना मदत करण्याचा निर्णय घेण्यात आला आहे. एरोनेट स्टोर्समध्ये २८ वर्षांपूर्वी १५ लाख प्लास्टिक पेअर्स आणि ४० लाख रुपये भाडे असे सर्वांत मोठे पैकीच व
